# Домашнее задание. Урок 34. Git контроль версий

Блокнот создан в локальном репозитории и отправлен на GitHub.

По заданию нужно написать три функции:

1. принимает `a: int`, `b: int`, `length: int` и возвращает список длины `length`
   из равномерно распределённой случайной величины между `a` и `b`;
2. принимает список и строит частотный ряд в виде словаря;
3. считает минимум, среднее, медиану, моду и максимум для значений списка.

Функции используют только встроенные средства Python; из внешних модулей
подключается лишь `random` для генерации случайных чисел.

In [1]:
import random

# Фиксируем зерно генератора, чтобы результаты в блокноте воспроизводились
random.seed(34)

## Функция 1. Генерация списка равномерно распределённых значений

Значения целые, потому что дальше по ним строится частотный ряд и считается мода —
для непрерывной величины оба показателя смысла не имеют: каждое значение
встретилось бы ровно один раз.

`random.randint(a, b)` возвращает целое число из отрезка от `a` до `b` включительно,
причём все значения равновероятны — это и есть равномерное распределение.

In [2]:
def generate_uniform_list(a: int, b: int, length: int) -> list:
    """Вернуть список длины length из равномерно распределённых целых чисел от a до b.

    a      — нижняя граница диапазона (включительно)
    b      — верхняя граница диапазона (включительно)
    length — требуемая длина списка
    """
    if length < 0:
        raise ValueError("Длина списка не может быть отрицательной")
    if a > b:
        # если границы перепутали местами, меняем их, а не падаем с ошибкой
        a, b = b, a

    return [random.randint(a, b) for _ in range(length)]

In [3]:
# Проверка на небольшом списке
sample = generate_uniform_list(1, 6, 20)
print("Список:", sample)
print("Длина:", len(sample))
print("Минимум и максимум:", min(sample), max(sample))

Список: [5, 3, 5, 1, 2, 1, 4, 3, 1, 4, 3, 3, 1, 5, 5, 2, 1, 3, 3, 5]
Длина: 20
Минимум и максимум: 1 5


In [4]:
# Проверка граничных случаев
print("Пустой список:      ", generate_uniform_list(1, 10, 0))
print("Одно значение:      ", generate_uniform_list(5, 5, 4))
print("Границы наоборот:   ", generate_uniform_list(9, 3, 6))

Пустой список:       []
Одно значение:       [5, 5, 5, 5]
Границы наоборот:    [5, 5, 8, 7, 8, 9]


## Функция 2. Частотный ряд

Функция проходит по списку и накапливает количество вхождений каждого значения.
Метод `dict.get(value, 0)` возвращает текущий счётчик, а если значение встретилось
впервые — ноль, поэтому отдельная проверка на наличие ключа не нужна.

In [5]:
def build_frequency_table(values: list) -> dict:
    """Построить частотный ряд: сколько раз каждое значение встречается в списке.

    Например, для [1, 2, 3, 1, 2, 2] вернёт {1: 2, 2: 3, 3: 1}.
    """
    frequency = {}
    for value in values:
        frequency[value] = frequency.get(value, 0) + 1

    # сортируем по значению, чтобы результат было удобно читать
    return dict(sorted(frequency.items()))

In [6]:
# Пример из условия задания
example = [1, 2, 3, 1, 2, 2]
result = build_frequency_table(example)

print("Список:       ", example)
print("Частотный ряд:", result)
print("Совпадает с ожидаемым {1: 2, 2: 3, 3: 1}:", result == {1: 2, 2: 3, 3: 1})

Список:        [1, 2, 3, 1, 2, 2]
Частотный ряд: {1: 2, 2: 3, 3: 1}
Совпадает с ожидаемым {1: 2, 2: 3, 3: 1}: True


In [7]:
# На сгенерированном списке
freq = build_frequency_table(sample)
print("Список:       ", sample)
print("Частотный ряд:", freq)
print("Сумма частот должна равняться длине списка:",
      sum(freq.values()), "==", len(sample))

Список:        [5, 3, 5, 1, 2, 1, 4, 3, 1, 4, 3, 3, 1, 5, 5, 2, 1, 3, 3, 5]
Частотный ряд: {1: 5, 2: 2, 3: 6, 4: 2, 5: 5}
Сумма частот должна равняться длине списка: 20 == 20


## Функция 3. Описательные статистики

Минимум, максимум и среднее считаются напрямую. Медиана и мода требуют пояснения.

**Медиана** — значение, делящее упорядоченный ряд пополам. При нечётной длине
это средний элемент, при чётной — полусумма двух средних. Поэтому список
сначала сортируется.

**Мода** — самое часто встречающееся значение. Здесь переиспользуется
функция 2: частотный ряд уже содержит всё необходимое, остаётся выбрать
значение с наибольшей частотой. Мод может оказаться несколько, поэтому
функция возвращает список всех наиболее частых значений — так честнее,
чем молча отдавать первое попавшееся.

In [8]:
def describe_values(values: list) -> dict:
    """Посчитать минимум, среднее, медиану, моду и максимум для списка чисел."""
    if not values:
        raise ValueError("Список пуст, статистики посчитать нельзя")

    ordered = sorted(values)
    n = len(ordered)

    # медиана
    middle = n // 2
    if n % 2 == 1:
        median = ordered[middle]
    else:
        median = (ordered[middle - 1] + ordered[middle]) / 2

    # мода — берём из частотного ряда все значения с максимальной частотой
    frequency = build_frequency_table(values)
    max_count = max(frequency.values())
    mode = [value for value, count in frequency.items() if count == max_count]

    return {
        "min": min(ordered),
        "mean": sum(ordered) / n,
        "median": median,
        "mode": mode,
        "max": max(ordered),
    }

In [9]:
# Проверка на примере из условия
stats_example = describe_values([1, 2, 3, 1, 2, 2])
for name, value in stats_example.items():
    print(f"{name:7}: {value}")

min    : 1
mean   : 1.8333333333333333
median : 2.0
mode   : [2]
max    : 3


In [10]:
# Проверка на сгенерированном списке
stats = describe_values(sample)

print("Список:", sample)
print()
for name, value in stats.items():
    print(f"{name:7}: {value}")

Список: [5, 3, 5, 1, 2, 1, 4, 3, 1, 4, 3, 3, 1, 5, 5, 2, 1, 3, 3, 5]

min    : 1
mean   : 3.0
median : 3.0
mode   : [3]
max    : 5


In [11]:
# Случай с несколькими модами: 1 и 3 встречаются по два раза
print(describe_values([1, 1, 2, 3, 3]))

# Случай с чётной длиной: медиана считается как полусумма двух средних
print(describe_values([10, 20, 30, 40]))

{'min': 1, 'mean': 2.0, 'median': 2, 'mode': [1, 3], 'max': 3}
{'min': 10, 'mean': 25.0, 'median': 25.0, 'mode': [10, 20, 30, 40], 'max': 40}


## Совместная проверка всех трёх функций

Соберём цепочку: генерируем большой список, строим по нему частотный ряд
и считаем статистики.

На большом списке равномерность распределения видна по частотам: все значения
должны встречаться примерно одинаково часто.

In [12]:
big_list = generate_uniform_list(1, 6, 6000)

big_freq = build_frequency_table(big_list)
big_stats = describe_values(big_list)

print("Частотный ряд по 6000 значениям:")
for value, count in big_freq.items():
    share = count / len(big_list) * 100
    bar = "#" * int(share)
    print(f"  {value}: {count:5}  ({share:5.2f}%)  {bar}")

print()
print("Ожидаемая доля каждого значения при равномерном распределении: 16.67%")
print()
print("Статистики:")
for name, value in big_stats.items():
    print(f"  {name:7}: {value}")

Частотный ряд по 6000 значениям:
  1:   985  (16.42%)  ################
  2:   985  (16.42%)  ################
  3:  1015  (16.92%)  ################
  4:  1000  (16.67%)  ################
  5:  1002  (16.70%)  ################
  6:  1013  (16.88%)  ################

Ожидаемая доля каждого значения при равномерном распределении: 16.67%

Статистики:
  min    : 1
  mean   : 3.514666666666667
  median : 4.0
  mode   : [3]
  max    : 6


Частоты действительно близки к ожидаемым 16,67% на каждое из шести значений,
а среднее близко к 3,5 — теоретическому среднему равномерного распределения
на отрезке от 1 до 6. Это подтверждает, что первая функция выдаёт именно
равномерно распределённую величину.

## Выводы

Все три функции написаны и проверены как на примере из условия задания,
так и на сгенерированных данных, включая граничные случаи: пустой список,
одинаковые границы диапазона, переставленные местами границы, чётная длина
списка и наличие нескольких мод.

Третья функция переиспользует вторую для расчёта моды — частотный ряд уже
содержит нужные данные, и считать их заново не имеет смысла.